# Week 4: Chunking Experiments

**Target Limitations:**
- LIM-001: Text order mismatch (79.4% multi-column)
- LIM-005: No category filtering
- LIM-006: Common term contamination
- LIM-008: Retrieval gap for specific facts

**Strategies:**
- A (baseline): Recursive 1000/200
- B1 (small): Recursive 500/100
- B2 (large): Recursive 1500/300
- C3 (layout-aware): unstructured/docling/marker — 2hr time box
- C2 (custom sep): fallback if C3 fails

## 1. Setup

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
import time

load_dotenv()

PDF_DIR = Path("../data/raw_pdfs")
CHROMA_DIR = Path("../data/chroma_db")

# Test queries from Week 3
TEST_QUERIES = [
    {"query": "정수기 필터 교체는 어떻게 하나요?", "expected_category": "waterpurifier"},
    {"query": "공기청정기 필터 청소 방법 알려주세요", "expected_category": "airpurifier"},
    {"query": "청소기 배터리 충전 시간은 얼마나 되나요?", "expected_category": "unknown"},  # vacuum typo
    {"query": "Wi-Fi 연결이 안될 때 어떻게 해야 하나요?", "expected_category": None},  # cross-category
]

print(f"PDF directory: {PDF_DIR}")
print(f"PDFs: {list(PDF_DIR.glob('*.pdf'))}")

## 2. Chunking Strategies

In [ ]:
from langchain_community.document_loaders import PDFPlumberLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import re

def parse_filename(filename: str) -> dict:
    """Parse metadata from filename."""
    name = Path(filename).stem.lower()
    
    categories = ["waterpurifier", "airpurifier", "vacuumcleaner", "vaccumcleaner"]
    category = "unknown"
    for cat in categories:
        if cat in name:
            category = cat
            break
    
    complexity = "complex" if "complex" in name else "simple"
    
    return {"category": category, "complexity": complexity}

def chunk_with_strategy(pdf_path: Path, chunk_size: int, overlap: int) -> list[dict]:
    """Chunk a PDF with given parameters."""
    loader = PDFPlumberLoader(str(pdf_path))
    pages = loader.load()
    
    meta = parse_filename(pdf_path.name)
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separators=["\n\n", "\n", "。", ".", " ", ""]
    )
    
    all_chunks = []
    for page in pages:
        page_num = page.metadata.get("page", 0) + 1
        chunks = splitter.split_text(page.page_content)
        
        for idx, chunk_text in enumerate(chunks):
            chunk_id = f"{meta['category']}_{meta['complexity']}_p{page_num:03d}_c{idx:03d}"
            all_chunks.append({
                "text": chunk_text,
                "metadata": {
                    "source": pdf_path.name,
                    "category": meta["category"],
                    "complexity": meta["complexity"],
                    "page": page_num,
                    "chunk_id": chunk_id,
                    "chunk_index": idx,
                    "char_count": len(chunk_text),
                }
            })
    
    return all_chunks

## 3. Strategy A — Baseline (1000/200)

In [ ]:
# Strategy A: Baseline (already exists in chroma_db)
# Load existing vector store for comparison

from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore_A = Chroma(
    persist_directory=str(CHROMA_DIR),
    embedding_function=embeddings,
    collection_name="lg_manuals",
)

print(f"Strategy A (baseline): {vectorstore_A._collection.count()} documents")

In [ ]:
# Test retrieval with Strategy A
retriever_A = vectorstore_A.as_retriever(search_kwargs={"k": 5})

def evaluate_retrieval(retriever, queries):
    """Evaluate retrieval accuracy."""
    results = []
    for q in queries:
        docs = retriever.invoke(q["query"])
        correct = 0
        for doc in docs:
            cat = doc.metadata.get("category", "unknown")
            if q["expected_category"] is None:  # cross-category OK
                correct += 1
            elif cat == q["expected_category"]:
                correct += 1
        results.append({
            "query": q["query"][:30],
            "correct": correct,
            "total": len(docs),
            "accuracy": correct / len(docs) if docs else 0
        })
    return results

results_A = evaluate_retrieval(retriever_A, TEST_QUERIES)
print("Strategy A (baseline) Retrieval:")
for r in results_A:
    print(f"  {r['query']}... {r['correct']}/{r['total']} ({r['accuracy']:.0%})")

## 4. Strategy B1 — Small Chunks (500/100)

In [ ]:
# Strategy B1: Smaller chunks
CHUNK_SIZE_B1 = 500
CHUNK_OVERLAP_B1 = 100

all_chunks_B1 = []
for pdf_path in sorted(PDF_DIR.glob("*.pdf")):
    chunks = chunk_with_strategy(pdf_path, CHUNK_SIZE_B1, CHUNK_OVERLAP_B1)
    all_chunks_B1.extend(chunks)
    print(f"  {pdf_path.name}: {len(chunks)} chunks")

print(f"\nStrategy B1 total: {len(all_chunks_B1)} chunks")
avg_size = sum(c['metadata']['char_count'] for c in all_chunks_B1) / len(all_chunks_B1)
print(f"Avg chunk size: {avg_size:.0f} chars")

In [ ]:
# Create temp vector store for B1
from langchain_core.documents import Document

docs_B1 = [Document(page_content=c["text"], metadata=c["metadata"]) for c in all_chunks_B1]

vectorstore_B1 = Chroma.from_documents(
    documents=docs_B1,
    embedding=embeddings,
    collection_name="lg_manuals_B1",
)

retriever_B1 = vectorstore_B1.as_retriever(search_kwargs={"k": 5})
results_B1 = evaluate_retrieval(retriever_B1, TEST_QUERIES)

print("Strategy B1 (500/100) Retrieval:")
for r in results_B1:
    print(f"  {r['query']}... {r['correct']}/{r['total']} ({r['accuracy']:.0%})")

## 5. Strategy B2 — Large Chunks (1500/300)

In [ ]:
# Strategy B2: Larger chunks
CHUNK_SIZE_B2 = 1500
CHUNK_OVERLAP_B2 = 300

all_chunks_B2 = []
for pdf_path in sorted(PDF_DIR.glob("*.pdf")):
    chunks = chunk_with_strategy(pdf_path, CHUNK_SIZE_B2, CHUNK_OVERLAP_B2)
    all_chunks_B2.extend(chunks)
    print(f"  {pdf_path.name}: {len(chunks)} chunks")

print(f"\nStrategy B2 total: {len(all_chunks_B2)} chunks")
avg_size = sum(c['metadata']['char_count'] for c in all_chunks_B2) / len(all_chunks_B2)
print(f"Avg chunk size: {avg_size:.0f} chars")

In [ ]:
# Create temp vector store for B2
docs_B2 = [Document(page_content=c["text"], metadata=c["metadata"]) for c in all_chunks_B2]

vectorstore_B2 = Chroma.from_documents(
    documents=docs_B2,
    embedding=embeddings,
    collection_name="lg_manuals_B2",
)

retriever_B2 = vectorstore_B2.as_retriever(search_kwargs={"k": 5})
results_B2 = evaluate_retrieval(retriever_B2, TEST_QUERIES)

print("Strategy B2 (1500/300) Retrieval:")
for r in results_B2:
    print(f"  {r['query']}... {r['correct']}/{r['total']} ({r['accuracy']:.0%})")

## 6. Strategy C3 — Layout-Aware Parsing

**Time box: 2 hours**

**Evaluation order:**
1. `unstructured` — partition_pdf with strategy="hi_res"
2. `docling` — IBM's document understanding
3. `marker-pdf` — multi-column aware, outputs markdown

**Goal:** Address LIM-001 (79.4% multi-column pages)

**Decision criteria:**
- Text order preserved in multi-column pages
- Processing time reasonable
- Korean text quality maintained

In [ ]:
# Try unstructured first
try:
    from unstructured.partition.pdf import partition_pdf
    UNSTRUCTURED_AVAILABLE = True
    print("unstructured available")
except ImportError:
    UNSTRUCTURED_AVAILABLE = False
    print("unstructured not available — install with: uv add unstructured[pdf]")

In [ ]:
# Test unstructured on sample PDF (if available)
if UNSTRUCTURED_AVAILABLE:
    sample_pdf = list(PDF_DIR.glob("waterpurifier_complex.pdf"))[0]
    print(f"Testing unstructured on: {sample_pdf.name}")
    
    start_time = time.time()
    elements = partition_pdf(
        filename=str(sample_pdf),
        strategy="fast",  # Start with fast, upgrade to hi_res if needed
    )
    elapsed = time.time() - start_time
    
    print(f"Time: {elapsed:.1f}s")
    print(f"Elements: {len(elements)}")
    print(f"Element types: {set(type(e).__name__ for e in elements)}")
    
    # Show sample elements
    for e in elements[:5]:
        print(f"  [{type(e).__name__}] {str(e)[:100]}...")
else:
    print("Skipping unstructured test")

In [ ]:
# Check multi-column handling with hi_res strategy
if UNSTRUCTURED_AVAILABLE:
    # Test specific page known to have multi-column (page 11 from Week 3)
    print("Testing hi_res strategy for multi-column...")
    print("(This may take longer)")
    
    start_time = time.time()
    elements_hires = partition_pdf(
        filename=str(sample_pdf),
        strategy="hi_res",
        infer_table_structure=True,
    )
    elapsed = time.time() - start_time
    
    print(f"Time (hi_res): {elapsed:.1f}s")
    print(f"Elements: {len(elements_hires)}")
else:
    print("Skipping hi_res test")

### C3 Option 2: docling

IBM's document understanding. Install: `uv add docling`

In [ ]:
# Try docling
try:
    from docling.document_converter import DocumentConverter
    DOCLING_AVAILABLE = True
    print("docling available")
except ImportError:
    DOCLING_AVAILABLE = False
    print("docling not available — install with: uv add docling")

In [ ]:
# Test docling if available
if DOCLING_AVAILABLE:
    sample_pdf = list(PDF_DIR.glob("waterpurifier_complex.pdf"))[0]
    print(f"Testing docling on: {sample_pdf.name}")
    
    start_time = time.time()
    converter = DocumentConverter()
    result = converter.convert(str(sample_pdf))
    elapsed = time.time() - start_time
    
    print(f"Time: {elapsed:.1f}s")
    print(f"Document: {result.document}")
else:
    print("Skipping docling test")

### C3 Option 3: marker-pdf

Multi-column aware PDF parser. Install: `uv add marker-pdf`

In [ ]:
# Try marker-pdf
try:
    from marker.converters.pdf import PdfConverter
    from marker.models import create_model_dict
    MARKER_AVAILABLE = True
    print("marker-pdf available")
except ImportError:
    MARKER_AVAILABLE = False
    print("marker-pdf not available — install with: uv add marker-pdf")

In [ ]:
# Test marker-pdf if available
if MARKER_AVAILABLE:
    sample_pdf = list(PDF_DIR.glob("waterpurifier_complex.pdf"))[0]
    print(f"Testing marker-pdf on: {sample_pdf.name}")
    
    start_time = time.time()
    model_dict = create_model_dict()
    converter = PdfConverter(artifact_dict=model_dict)
    result = converter(str(sample_pdf))
    elapsed = time.time() - start_time
    
    print(f"Time: {elapsed:.1f}s")
    print(f"Markdown output length: {len(result.markdown)} chars")
    print(f"\\nSample output (first 1000 chars):")
    print("-" * 50)
    print(result.markdown[:1000])
else:
    print("Skipping marker-pdf test")

## 7. Strategy C2 — Custom Separators (Fallback)

If C3 (layout-aware) fails or takes too long, use custom separators:
- Split by section headers
- Split by step numbers
- Preserve table boundaries

In [ ]:
# Custom separators based on Korean manual patterns
KOREAN_SEPARATORS = [
    "\n\n\n",           # Triple newline (section break)
    "\n\n",             # Double newline (paragraph break)
    r"\n\d+\s",         # Step numbers ("1 ", "2 ")
    "\n•\s",            # Bullet points
    "\n- ",             # Dash lists
    "\n",               # Single newline
    "。",               # Korean period
    ".",                # Period
    " ",                # Space
    "",                 # Character
]

def chunk_with_custom_sep(pdf_path: Path, chunk_size: int = 1000, overlap: int = 200) -> list[dict]:
    """Chunk with Korean-specific separators."""
    loader = PDFPlumberLoader(str(pdf_path))
    pages = loader.load()
    
    meta = parse_filename(pdf_path.name)
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separators=KOREAN_SEPARATORS,
    )
    
    all_chunks = []
    for page in pages:
        page_num = page.metadata.get("page", 0) + 1
        chunks = splitter.split_text(page.page_content)
        
        for idx, chunk_text in enumerate(chunks):
            chunk_id = f"{meta['category']}_{meta['complexity']}_p{page_num:03d}_c{idx:03d}"
            all_chunks.append({
                "text": chunk_text,
                "metadata": {
                    "source": pdf_path.name,
                    "category": meta["category"],
                    "complexity": meta["complexity"],
                    "page": page_num,
                    "chunk_id": chunk_id,
                    "chunk_index": idx,
                    "char_count": len(chunk_text),
                }
            })
    
    return all_chunks

In [ ]:
# Test C2 on all PDFs
all_chunks_C2 = []
for pdf_path in sorted(PDF_DIR.glob("*.pdf")):
    chunks = chunk_with_custom_sep(pdf_path)
    all_chunks_C2.extend(chunks)
    print(f"  {pdf_path.name}: {len(chunks)} chunks")

print(f"\nStrategy C2 total: {len(all_chunks_C2)} chunks")
avg_size = sum(c['metadata']['char_count'] for c in all_chunks_C2) / len(all_chunks_C2)
print(f"Avg chunk size: {avg_size:.0f} chars")

In [ ]:
# Create vector store for C2
docs_C2 = [Document(page_content=c["text"], metadata=c["metadata"]) for c in all_chunks_C2]

vectorstore_C2 = Chroma.from_documents(
    documents=docs_C2,
    embedding=embeddings,
    collection_name="lg_manuals_C2",
)

retriever_C2 = vectorstore_C2.as_retriever(search_kwargs={"k": 5})
results_C2 = evaluate_retrieval(retriever_C2, TEST_QUERIES)

print("Strategy C2 (custom sep) Retrieval:")
for r in results_C2:
    print(f"  {r['query']}... {r['correct']}/{r['total']} ({r['accuracy']:.0%})")

## 8. Comparison Summary

In [ ]:
# Build comparison table
def avg_accuracy(results):
    return sum(r['accuracy'] for r in results) / len(results)

comparison = [
    {"strategy": "A (baseline 1000/200)", "chunks": vectorstore_A._collection.count(), "avg_accuracy": avg_accuracy(results_A)},
    {"strategy": "B1 (small 500/100)", "chunks": len(all_chunks_B1), "avg_accuracy": avg_accuracy(results_B1)},
    {"strategy": "B2 (large 1500/300)", "chunks": len(all_chunks_B2), "avg_accuracy": avg_accuracy(results_B2)},
    {"strategy": "C2 (custom sep)", "chunks": len(all_chunks_C2), "avg_accuracy": avg_accuracy(results_C2)},
]

print(f"{'Strategy':<25} {'Chunks':<10} {'Avg Accuracy':<15}")
print("-" * 50)
for c in comparison:
    print(f"{c['strategy']:<25} {c['chunks']:<10} {c['avg_accuracy']:.0%}")

## 9. Qualitative Analysis: LIM-005/008 Queries

In [ ]:
# Compare retrieval for LIM-005 query across strategies
query_005 = "정수기 필터 교체는 어떻게 하나요?"

print(f"Query: {query_005}")
print("=" * 70)

for name, retriever in [("A", retriever_A), ("B1", retriever_B1), ("B2", retriever_B2), ("C2", retriever_C2)]:
    docs = retriever.invoke(query_005)
    categories = [d.metadata.get('category', '?') for d in docs]
    correct = sum(1 for c in categories if c == 'waterpurifier')
    print(f"\n[{name}] {correct}/5 correct")
    print(f"    Categories: {categories}")

In [ ]:
# Compare retrieval for LIM-008 query across strategies
query_008 = "청소기 배터리 충전 시간은 얼마나 되나요?"

print(f"Query: {query_008}")
print("=" * 70)

for name, retriever in [("A", retriever_A), ("B1", retriever_B1), ("B2", retriever_B2), ("C2", retriever_C2)]:
    docs = retriever.invoke(query_008)
    
    print(f"\n[{name}]")
    for i, doc in enumerate(docs):
        has_charging = '충전' in doc.page_content
        has_time = '시간' in doc.page_content or '분' in doc.page_content
        cat = doc.metadata.get('category', '?')
        print(f"  [{i+1}] {cat} | 충전: {'✓' if has_charging else '✗'} | 시간/분: {'✓' if has_time else '✗'}")

## 10. Notes

### Findings
- (fill after running)

### C3 (Layout-aware) Status
- (fill after running)

### Recommendation
- (fill after running)